# MODELS CREATION AND TRAINING

Down below you can find the code for the models creation for the socialized learning part. As in the paper, we are gonna create and train 5 agents, each agent will "become expert" in 2 classes from the CIFAR-10 dataset.

In [16]:
# import libraries
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import random
import torch.nn.functional as F
import platform as pl


#fix seeds
np.random.seed(0)
torch.manual_seed(0)
random.seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


We are going to use the model structure provided in HW3

In [17]:
def create_model():
    '''
    Create a simple CNN model for CIFAR10 dataset
    '''

    model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=(3, 3), stride=1, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AvgPool2d(kernel_size=2, stride=2),
        nn.Dropout(p=0.1),

        nn.Conv2d(32, 64, kernel_size=(3, 3), stride=1, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.AvgPool2d(kernel_size=2, stride=2),
        nn.Dropout(p=0.1),

        nn.Conv2d(64, 64, kernel_size=(3, 3), stride=1, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((1,1)),
        nn.Flatten(),
        nn.Dropout(p=0.1),
        
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Dropout(p=0.1),
        
        nn.Linear(32, 10)
    )
    
    return model

# DATASET PREPARATION

In [18]:
# Load the dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())

'''
Add your code below
'''
images = np.stack([train_dataset[i][0] for i in range(len(train_dataset))])

# Compute the mean and standard deviation for each channel
mean = images.mean(axis=(0, 2, 3))
std = images.std(axis=(0, 2, 3))

print("Mean: ", np.round(mean, 4))
print("Std: ", np.round(std, 4))

Files already downloaded and verified
Mean:  [0.4914 0.4822 0.4465]
Std:  [0.247  0.2435 0.2616]


In [19]:
# Filter dataset to exclude classes
def filter_dataset(dataset: torch.utils.data.Dataset, target_classes: list) -> torch.utils.data.Subset:
    """
    # Dataset Filtering for Target and Non-Target Classes

    The **filter_dataset** function filters a dataset based on the specified target classes
    Args:
        - dataset (torch.utils.data.Dataset): The dataset to be filtered.
        - target_classes (list): A list of class labels to be considered as target or non-target classes.

    Returns:
        - torch.utils.data.Subset: A subset of the original dataset containing only the target
    """
    labels = np.array([label for _, label in dataset])
    indices = [i for i, label in enumerate(labels) if label in target_classes]

    filtered_dataset = torch.utils.data.Subset(dataset, indices)
    return filtered_dataset

We are going to split the datasets to make our 5 agents expert in different classes

In [20]:
# Define the augmentations for the training set
cifar_transforms = transforms.Compose([
    transforms.ToTensor(),                    # Convert the image to a PyTorch tensor
    transforms.Normalize(mean, std),          # Normalize the image channel
])

# Load the CIFAR-10 dataset with the appropriate transforms
train_dataset = datasets.CIFAR10(root="data", train=True, transform=cifar_transforms, download=True)  
test_dataset = datasets.CIFAR10(root="data", train=False, transform=cifar_transforms, download=True)

train_datasets = []
test_datasets = []
val_datasets = []
classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]

# creation of train and test parts
for idx in range(5):
    train_datasets.append(filter_dataset(train_dataset, classes[idx]))
    test_datasets.append(filter_dataset(test_dataset, classes[idx]))
    # creation of the validation
    val_dataset, test_datasets[idx] = torch.utils.data.random_split(test_datasets[idx], [400, 1600])
    val_datasets.append(val_dataset)

Files already downloaded and verified
Files already downloaded and verified


In [21]:
train_loaders = []
test_loaders = []
val_loaders = []
batch_size = 512

for idx in range(5):
    # Create the DataLoaders
    train_loaders.append(DataLoader(train_datasets[idx], batch_size = batch_size, shuffle=True))
    val_loaders.append(DataLoader(val_datasets[idx], batch_size = batch_size, shuffle=False))
    test_loaders.append(DataLoader(test_datasets[idx], batch_size = batch_size, shuffle=False))

# MODELS CREATION AND TRAINING

In [22]:
# We had to add this check cause we are also using ARM systems
if pl.system() == "Darwin":
    device = torch.device("mps" if torch.mps.is_available() else "cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

agents = []
for idx in range(5):
    model = create_model()
    model.load_state_dict(torch.load('checkpoint/model_weights.pth', weights_only=True))  
    model.to(device)

    # initialize the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)
    num_epochs = 20

    # Placeholder for storing losses for each epoch
    losses = []
    losses_val = []

    # Training the model
    for epoch in range(num_epochs):

        ######### TRAINING ##########
        model.train()
        running_loss = 0  # To track loss for this epoch

        # Using tqdm for the progress bar
        loop = tqdm(enumerate(train_loaders[idx]), total=len(train_loaders[idx]), leave=True)
        
        for batch_idx, (data, targets) in loop:
            # Get data to cuda if possible
            data = data.to(device=device)
            targets = targets.to(device=device)

            # Forward pass
            scores = model(data)
            loss = criterion(scores, targets)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient descent step
            optimizer.step()

            # Accumulate loss
            running_loss += loss.item()

            # Update progress bar with loss and epoch information
            loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
            loop.set_postfix(loss=loss.item())

        # Calculate average loss for the epoch
        avg_loss = running_loss / len(train_loaders[idx])
        losses.append(avg_loss)

        #scheduler 
        scheduler.step(avg_loss)

        # Print loss for this epoch
        tqdm.write(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_loss:.4f}")

        ####### VALIDATION ########
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for data, targets in val_loaders[idx]:
                data = data.to(device=device)
                targets = targets.to(device=device)

                scores = model(data)
                loss = criterion(scores, targets)
                val_loss += loss.item()
            # Calculate average loss for the epoch
            avg_val_loss = val_loss / len(val_loaders[idx])
            losses_val.append(avg_val_loss)
            print(f"Validation Loss: {avg_val_loss:.4f}")
            # if avg val_loss is better than the one before, save the model
            if epoch == 0:
                # create directory if not exist
                os.makedirs("checkpoint", exist_ok=True)
                best_loss = avg_val_loss
                torch.save(model.state_dict(), "checkpoint/agent_"+str(idx)+"_trained_model.pth")
            elif avg_val_loss < best_loss:
                best_loss = avg_val_loss
                torch.save(model.state_dict(), "checkpoint/agent_"+str(idx)+"_trained_model.pth")

c:\Users\radul\anaconda3\envs\AML_env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch [1/20]: 100%|██████████| 20/20 [00:05<00:00,  3.46it/s, loss=1.46]


Epoch [1/20], Average Loss: 1.8780
Validation Loss: 1.6085


Epoch [2/20]: 100%|██████████| 20/20 [00:05<00:00,  3.52it/s, loss=0.768]


Epoch [2/20], Average Loss: 1.0586
Validation Loss: 0.8479


Epoch [3/20]: 100%|██████████| 20/20 [00:05<00:00,  3.61it/s, loss=0.433]


...

Epoch [18/20], Average Loss: 0.1446
Validation Loss: 0.1797


Epoch [19/20]: 100%|██████████| 20/20 [00:05<00:00,  3.74it/s, loss=0.15] 


Epoch [19/20], Average Loss: 0.1459
Validation Loss: 0.2039


Epoch [20/20]: 100%|██████████| 20/20 [00:05<00:00,  3.69it/s, loss=0.143] 


Epoch [20/20], Average Loss: 0.1393
Validation Loss: 0.1605


# EVALUATION OF THE TRAINED AGENTS

In [23]:
# accuracy 
def accuracy (model, loader):
    '''
    Function to calculate the accuracy of the model on the test set
    '''
    correct = 0
    total = 0
    for data, targets in loader:
        data = data.to(device=device)
        targets = targets.to(device=device)
        scores = model(data)
        _, predictions = scores.max(1)
        correct += (predictions == targets).sum()
        total += targets.shape[0]
    return correct / total

In [25]:
for idx in range(5):
    model = create_model()
    model.load_state_dict(torch.load('checkpoint/agent_'+str(idx)+'_trained_model.pth', weights_only=True))
    model.eval()
    model.to(device)

    # Calculate accuracy on the test set 
    test_accuracy = accuracy(model, test_loaders[idx])
    print(f"Your Agent N°{str(idx)} Test Accuracy : {100* test_accuracy:.4f}")

Your Agent N°0 Test Accuracy : 93.8125
Your Agent N°1 Test Accuracy : 83.3125
Your Agent N°2 Test Accuracy : 87.4375
Your Agent N°3 Test Accuracy : 95.3125
Your Agent N°4 Test Accuracy : 94.1250
